# ANALÍTICA EMPRESARIAL INTEGRADA
## LABORATORIO N.° 11 — Analítica Prescriptiva: Optimización

**Semana 11 · TECSUP 2026-II**  
**Docente:** Pilar Rocío Sayán Mejía  
**Dataset real:** Online Retail II — UCI Machine Learning Repository  
**Herramienta:** PuLP

### Pregunta de negocio
**¿Cómo debe distribuirse una capacidad limitada de inventario entre productos para maximizar el valor potencial de venta sin descuidar niveles mínimos de servicio?**

> El objetivo no es solo obtener `Optimal`. Debe demostrarse cuánto mejora la decisión frente a una regla intuitiva.

## Identificación
- Equipo:
- Sección: 6C28
- Integrantes: Piero Hideki Furushio Casanave
- Repositorio:
- Uso de IA declarado: Sí. Se utilizó un asistente de IA (Qwen) para la generación de código de los retos de optimización y la redacción de las respuestas teóricas. El estudiante validó y ajustó todos los resultados antes de la entrega.


In [1]:
# Instala en Colab las librerías que no vienen por defecto; la versión de PuLP queda fija para repetir resultados
%pip install -q "ucimlrepo>=0.0.7" "pulp==3.3.0" plotly openpyxl

In [2]:
# Librerías: pandas y numpy para los datos, PuLP para optimizar y plotly para graficar
import sys
import numpy as np
import pandas as pd
import plotly.express as px
import pulp as pl
from ucimlrepo import fetch_ucirepo
# Se registran las versiones para que otra persona pueda repetir el resultado
print("Python:", sys.version.split()[0], "· pandas:", pd.__version__, "· PuLP:", pl.__version__)
# Permite ver hasta 50 columnas al imprimir tablas
pd.set_option("display.max_columns", 50)

Python: 3.13.18 · pandas: 2.2.3 · PuLP: 3.3.0


# 1. Descarga de datos reales y trazabilidad

In [3]:
# API UCI; si falla, usar el ZIP oficial. La carga manual queda como último recurso.
from io import BytesIO
from urllib.request import urlopen
from zipfile import ZipFile
from pathlib import Path

try:
    retail = fetch_ucirepo(id=502)
    df = retail.data.original.copy()
    fuente = "API ucimlrepo, UCI ID 502"
except Exception as exc:
    print("API no disponible:", type(exc).__name__)
    try:
        url = "https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip"
        with urlopen(url, timeout=120) as response:
            with ZipFile(BytesIO(response.read())) as z:
                name = next(n for n in z.namelist() if n.lower().endswith(".xlsx"))
                df = pd.concat(pd.read_excel(BytesIO(z.read(name)), sheet_name=None).values(), ignore_index=True)
        fuente = "ZIP oficial UCI, ID 502"
    except Exception:
        local = Path("online_retail_II.xlsx")
        if not local.exists():
            raise FileNotFoundError("Descargue online_retail_II.xlsx desde UCI y súbalo a Colab.")
        df = pd.concat(pd.read_excel(local, sheet_name=None).values(), ignore_index=True)
        fuente = "Archivo local descargado de UCI"
print("Fuente:", fuente, "· ejecución UTC:", pd.Timestamp.now(tz="UTC").isoformat())
print("Dimensión:", df.shape, "· columnas:", df.columns.tolist())
display(df.head())

API no disponible: DatasetNotFoundError
Fuente: ZIP oficial UCI, ID 502 · ejecución UTC: 2026-10-29T04:33:30.344044+00:00
Dimensión: (1067371, 8) · columnas: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


## Normalización de nombres

In [4]:
# El archivo trae nombres de columna distintos según la fuente (Invoice o InvoiceNo, Price o UnitPrice); esta función busca cuál se usa
def find_col(columns, candidates):
    normalized = {str(c).lower().replace(" ", "").replace("_",""): c for c in columns}
    for cand in candidates:
        key = cand.lower().replace(" ", "").replace("_","")
        if key in normalized:
            return normalized[key]
    return None

# Se traduce cada columna a un nombre único que usa el resto del notebook
mapping = {
    "InvoiceNo": find_col(df.columns, ["InvoiceNo", "Invoice"]),
    "StockCode": find_col(df.columns, ["StockCode"]),
    "Description": find_col(df.columns, ["Description"]),
    "Quantity": find_col(df.columns, ["Quantity"]),
    "InvoiceDate": find_col(df.columns, ["InvoiceDate"]),
    "UnitPrice": find_col(df.columns, ["UnitPrice", "Price"]),
    "Country": find_col(df.columns, ["Country"]),
}

# Si falta una columna indispensable, se detiene aquí en lugar de fallar más adelante
required = ["InvoiceNo","StockCode","Quantity","InvoiceDate","UnitPrice","Country"]
missing = [k for k in required if mapping[k] is None]
if missing:
    raise ValueError(f"No se identificaron columnas obligatorias: {missing}")

# Se renombran las columnas y se muestra la correspondencia encontrada
df = df.rename(columns={v:k for k,v in mapping.items() if v is not None})
print(mapping)

{'InvoiceNo': 'Invoice', 'StockCode': 'StockCode', 'Description': 'Description', 'Quantity': 'Quantity', 'InvoiceDate': 'InvoiceDate', 'UnitPrice': 'Price', 'Country': 'Country'}


# 2. Control de calidad

In [5]:
# Se convierten fecha, cantidad y precio a su tipo correcto; lo que no se pueda convertir queda vacío
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"], errors="coerce")
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["UnitPrice"] = pd.to_numeric(df["UnitPrice"], errors="coerce")
# Se quitan los registros sin código de producto y se limpian espacios
df = df[df["StockCode"].notna()].copy()
df["StockCode"] = df["StockCode"].astype(str).str.strip()
# Las facturas que empiezan con C son cancelaciones: restan demanda y no deben contarse como ventas
cancel = df["InvoiceNo"].astype(str).str.upper().str.startswith("C")
# Regla de limpieza: solo ventas válidas (cantidad y precio positivos, con fecha, producto y país)
clean = df.loc[(~cancel) & df["InvoiceDate"].notna() &
               df["StockCode"].ne("") & df["Country"].notna() &
               (df["Quantity"] > 0) & (df["UnitPrice"] > 0)].copy()
# Se conserva el Reino Unido, el mercado principal del archivo
uk = clean[clean["Country"].astype(str).str.upper().eq("UNITED KINGDOM")].copy()
# Se registra cuántas cancelaciones había y cuántas filas válidas quedaron
print("Cancelaciones:", int(cancel.sum()), "· filas válidas UK:", len(uk))
print("Periodo:", uk["InvoiceDate"].min(), "→", uk["InvoiceDate"].max())

Cancelaciones: 19494 · filas válidas UK: 958501
Periodo: 2009-12-01 07:45:00 → 2011-12-09 12:49:00


### Pregunta 1
¿Por qué es incorrecto usar directamente la suma de Quantity sin revisar cancelaciones y precios?

**Respuesta:** Es incorrecto porque el dataset contiene facturas de cancelación (identificadas porque su Invoice comienza con "C"), las cuales tienen cantidades negativas. Sumar directamente Quantity restaría estas devoluciones, subestimando la demanda real. Además, pueden existir precios negativos o cero por errores de registro, lo que distorsionaría gravemente el cálculo del valor potencial de venta.

# 3. Construcción de parámetros reales

In [6]:
# Últimas 12 semanas COMPLETAS: lunes a domingo.
# El archivo termina el 9-dic-2011 (viernes); se excluye esa semana parcial.
last_obs = uk["InvoiceDate"].max().normalize()
last_complete_sunday = last_obs - pd.Timedelta(days=last_obs.weekday() + 1)
first_monday = last_complete_sunday - pd.Timedelta(weeks=12) + pd.Timedelta(days=1)
recent = uk.loc[
    (uk["InvoiceDate"] >= first_monday) &
    (uk["InvoiceDate"] < last_complete_sunday + pd.Timedelta(days=1))
].copy()
recent["week"] = recent["InvoiceDate"].dt.to_period("W-SUN").dt.start_time
assert recent["week"].nunique() == 12, "Revisar cobertura de las 12 semanas."

weekly = recent.groupby(["StockCode", "week"], as_index=False).agg(
    units=("Quantity", "sum"), price=("UnitPrice", "median")
)
sku_params = weekly.groupby("StockCode").agg(
    weeks_present=("week", "nunique"),
    demand_total=("units", "sum")
).reset_index()
sku_params["weekly_demand"] = sku_params["demand_total"] / 12
sku_params["price"] = sku_params["StockCode"].map(
    recent.groupby("StockCode")["UnitPrice"].median()
)
eligible = sku_params.loc[sku_params["weeks_present"] >= 8].copy()
if len(eligible) < 10:
    eligible = sku_params.loc[sku_params["weeks_present"] >= 6].copy()
if len(eligible) < 10:
    raise ValueError("Menos de 10 SKU elegibles; revisar datos y regla de selección.")
top = eligible.sort_values(["demand_total", "StockCode"], ascending=[False, True]).head(10).copy()
top["demanda"] = np.maximum(np.ceil(top["weekly_demand"]), 1).astype(int)
parametros = top.set_index("StockCode")[["demanda", "price", "weeks_present", "demand_total"]].copy()
print("Ventana:", first_monday.date(), "a", last_complete_sunday.date())
display(parametros)
print("Demanda esperada total:", int(parametros["demanda"].sum()))

Ventana: 2011-09-12 a 2011-12-04


,demanda,price,weeks_present,demand_total
StockCode,,,,
22197,1809,0.85,12,21703
84077,1223,0.29,12,14676
85099B,1175,2.08,12,14098
22086,1080,2.95,12,12951
23084,959,2.08,12,11505
84879,860,1.69,12,10310
22952,765,0.55,12,9177
22577,746,0.29,12,8948
22578,693,0.29,12,8305


Demanda esperada total: 9982


### Pregunta 2
¿Qué significa una semana sin ventas observadas para un SKU y por qué no prueba demanda cero?

**Respuesta:** Una semana sin ventas observadas no prueba que la demanda sea cero; puede deberse a un quiebre de stock (stockout), retrasos en el registro de datos, o simplemente a la aleatoriedad de que ningún cliente haya comprado ese producto en esa semana específica, a pesar de que exista demanda latente.

# 4. Supuestos explícitos del escenario

In [7]:
# Supuestos académicos del caso (no vienen de UCI): capacidad, servicio mínimo y concentración máxima
CAPACITY_RATIO = 0.70
MIN_SERVICE_RATIO = 0.20
MAX_CONCENTRATION_RATIO = 0.35
# Se derivan los límites en unidades enteras a partir de la demanda estimada
demanda_total = int(parametros["demanda"].sum())
capacidad_total = int(np.floor(CAPACITY_RATIO * demanda_total))
limite_concentracion = int(np.floor(MAX_CONCENTRATION_RATIO * capacidad_total))
# Cada SKU debe recibir como mínimo el 20 % de su demanda (y al menos una unidad)
parametros["min_servicio"] = np.maximum(
    np.ceil(MIN_SERVICE_RATIO * parametros["demanda"]).astype(int), 1
)
# Se comprueba que el problema tiene solución: los mínimos caben en la capacidad y en el tope por SKU
assert parametros["min_servicio"].sum() <= capacidad_total
assert (parametros["min_servicio"] <= limite_concentracion).all()
# Se muestran la demanda y la capacidad resultantes, y la tabla de parámetros
print("Demanda:", demanda_total, "· capacidad:", capacidad_total)
display(parametros)

Demanda: 9982 · capacidad: 6987


,demanda,price,weeks_present,demand_total,min_servicio
StockCode,,,,,
22197,1809,0.85,12,21703,362
84077,1223,0.29,12,14676,245
85099B,1175,2.08,12,14098,235
22086,1080,2.95,12,12951,216
23084,959,2.08,12,11505,192
84879,860,1.69,12,10310,172
22952,765,0.55,12,9177,153
22577,746,0.29,12,8948,150
22578,693,0.29,12,8305,139


> **Importante:** 70 % de capacidad, 20 % de servicio mínimo y 35 % de concentración son supuestos académicos del caso. No provienen de UCI.

### Pregunta 3
Separe parámetros derivados de UCI y supuestos de operaciones.

**Respuesta:**

*   Derivados de UCI (Datos): demanda (semanal), price, weeks_present, demand_total.
*   Supuestos de operaciones (Negocio): CAPACITY_RATIO (70%), MIN_SERVICE_RATIO (20%), MAX_CONCENTRATION_RATIO (35%), capacidad_total, limite_concentracion y min_servicio.



# 5. Línea base intuitiva: asignación proporcional a la demanda

In [8]:
# Línea base intuitiva: reparte la capacidad en proporción a la demanda, respetando mínimos, demanda y tope por SKU
def proportional_allocation(params, capacity, min_col="min_servicio", max_conc=None):
    """Regla proporcional con mínimos, demanda y concentración: baseline factible."""
    # Tope de cada SKU: su demanda y, si existe, el límite de concentración
    upper = params["demanda"].astype(int).copy()
    if max_conc is not None:
        upper = upper.clip(upper=int(max_conc))
    # Piso de cada SKU: el servicio mínimo
    lower = params[min_col].astype(int).copy()
    # Si las reglas no permiten asignar toda la capacidad, se avisa en lugar de comparar algo injusto
    if (lower > upper).any() or lower.sum() > capacity or upper.sum() < capacity:
        raise ValueError("La política no permite asignar toda la capacidad.")
    # Se parte de los mínimos y se reparte el resto en proporción a la demanda
    alloc = lower.copy()
    remaining = int(capacity - alloc.sum())
    weights = params["demanda"].astype(float)
    # Se repite hasta repartir todas las unidades, respetando el espacio que le queda a cada SKU
    while remaining:
        room = upper - alloc
        active = room > 0
        if not active.any():
            break
        quota = remaining * weights[active] / weights[active].sum()
        add = np.minimum(np.floor(quota).astype(int), room[active])
        if add.sum() == 0:
            order = (quota - np.floor(quota)).sort_values(ascending=False).index
            for sku in order:
                if remaining == 0: break
                if room.loc[sku] > 0:
                    alloc.loc[sku] += 1
                    remaining -= 1
        else:
            alloc.loc[active] += add
            remaining -= int(add.sum())
    # Verificación final: se asignó exactamente la capacidad
    assert alloc.sum() == capacity
    return alloc

# Se aplica la regla y se calcula su valor potencial: es la referencia que PuLP debe superar
baseline = parametros.copy()
baseline["asignacion_intuitiva"] = proportional_allocation(
    parametros, capacidad_total, max_conc=limite_concentracion
)
baseline["valor_intuitivo"] = baseline["asignacion_intuitiva"] * baseline["price"]
display(baseline)
print("Unidades intuitivas:", int(baseline["asignacion_intuitiva"].sum()))
print("Valor potencial intuitivo (£):", round(baseline["valor_intuitivo"].sum(), 2))

,demanda,price,weeks_present,demand_total,min_servicio,asignacion_intuitiva,valor_intuitivo
StockCode,,,,,,,
22197,1809,0.85,12,21703,362,1266,1076.10
84077,1223,0.29,12,14676,245,857,248.53
85099B,1175,2.08,12,14098,235,823,1711.84
22086,1080,2.95,12,12951,216,756,2230.20
23084,959,2.08,12,11505,192,672,1397.76
84879,860,1.69,12,10310,172,601,1015.69
22952,765,0.55,12,9177,153,535,294.25
22577,746,0.29,12,8948,150,522,151.38
22578,693,0.29,12,8305,139,485,140.65


Unidades intuitivas: 6987
Valor potencial intuitivo (£): 8853.9


### Pregunta 4
¿Por qué la comparación con PuLP sería engañosa si la regla intuitiva excediera un límite?

**Respuesta:** Sería engañosa porque si la regla intuitiva excede un límite (como la capacidad total o la concentración máxima), se convertiría en una solución infactible. Comparar un óptimo matemático factible contra una regla que viola las restricciones del negocio es una comparación injusta y no refleja la realidad operativa.

# 6. Formulación con PuLP

In [9]:
# Modelo: maximizar el valor potencial de venta (precio × unidades asignadas)
model = pl.LpProblem("Asignacion_Inventario", pl.LpMaximize)
# Variable de decisión: unidades enteras (mayores o iguales a 0) que se asignan a cada SKU
x = {sku: pl.LpVariable(f"x_{i}", lowBound=0, cat=pl.LpInteger)
     for i, sku in enumerate(parametros.index)}
# Función objetivo
model += pl.lpSum(parametros.loc[sku, "price"] * x[sku] for sku in parametros.index)
# Restricción de capacidad: no se puede asignar más de lo que hay
model += pl.lpSum(x.values()) <= capacidad_total
# Por cada SKU: no más que su demanda, no menos que su servicio mínimo y no más que el tope de concentración
for sku in parametros.index:
    model += x[sku] <= int(parametros.loc[sku, "demanda"])
    model += x[sku] >= int(parametros.loc[sku, "min_servicio"])
    model += x[sku] <= limite_concentracion
# Con 10 SKU se esperan 10 variables y 31 restricciones (1 de capacidad y 3 por SKU)
print("Variables:", len(x), "· restricciones:", len(model.constraints))

Variables: 10 · restricciones: 31


### Pregunta 5
Explique por qué maximizar precio × unidades sin restricciones no responde la decisión de negocio.

**Respuesta:** Porque llevaría a asignar toda la capacidad disponible únicamente al producto de mayor precio hasta el infinito (o hasta su demanda), ignorando la necesidad estratégica de mantener un nivel de servicio mínimo para otros productos, la capacidad física limitada del almacén y la diversificación del portafolio que requiere el negocio.

# 7. Resolver el problema

In [10]:
# Se elige el solucionador CBC, que viene con PuLP, y se resuelve el modelo
solver = pl.PULP_CBC_CMD(msg=False)
status = model.solve(solver)

# El estado dice si hay solución (Optimal, Infeasible, Unbounded o Not Solved); se lee antes que los números
print("Estado:", pl.LpStatus[status])
print("Valor objetivo (£):", round(pl.value(model.objective), 2))

Estado: Optimal
Valor objetivo (£): 11478.03


In [11]:
# No se sigue si el solucionador no encontró un óptimo
if pl.LpStatus[status] != "Optimal":
    raise RuntimeError(
        "El modelo no alcanzó estado Optimal. Revise capacidad, mínimos y concentración."
    )

# Se guarda la asignación óptima por SKU y su valor potencial
opt = parametros.copy()
opt["asignacion_optima"] = [int(round(x[sku].value())) for sku in opt.index]
opt["valor_optimo"] = opt["asignacion_optima"] * opt["price"]

# Se muestran la tabla y los totales: las unidades asignadas deben igualar la capacidad
display(opt)
print("Unidades asignadas:", int(opt["asignacion_optima"].sum()))
print("Valor potencial óptimo (£):", round(opt["valor_optimo"].sum(), 2))

,demanda,price,weeks_present,demand_total,min_servicio,asignacion_optima,valor_optimo
StockCode,,,,,,,
22197,1809,0.85,12,21703,362,1554,1320.90
84077,1223,0.29,12,14676,245,245,71.05
85099B,1175,2.08,12,14098,235,1175,2444.00
22086,1080,2.95,12,12951,216,1080,3186.00
23084,959,2.08,12,11505,192,959,1994.72
84879,860,1.69,12,10310,172,860,1453.40
22952,765,0.55,12,9177,153,153,84.15
22577,746,0.29,12,8948,150,150,43.50
22578,693,0.29,12,8305,139,139,40.31


Unidades asignadas: 6987
Valor potencial óptimo (£): 11478.03


### Pregunta 6
¿Qué garantiza el estado Optimal y qué no garantiza respecto de la demanda y los costos?

**Respuesta:** El estado Optimal garantiza que la solución es la mejor posible dentro del modelo matemático y sus restricciones definidas. No garantiza que los pronósticos de demanda sean exactos, ni considera costos no modelados (como costo de mantener inventario, costo de pedido o penalizaciones por quiebre de stock). Optimiza el "valor potencial de venta", no necesariamente la utilidad neta real.

# 8. Comparación: intuitivo vs óptimo

In [12]:
# Se juntan en una tabla la asignación intuitiva y la óptima, por SKU
comparison = parametros[["demanda","price"]].copy()
comparison["intuitivo"] = baseline["asignacion_intuitiva"]
comparison["optimo"] = opt["asignacion_optima"]
# Diferencia de unidades: positivo significa que el modelo asigna más que la regla intuitiva
comparison["dif_unidades"] = comparison["optimo"] - comparison["intuitivo"]
comparison["valor_intuitivo"] = comparison["intuitivo"] * comparison["price"]
comparison["valor_optimo"] = comparison["optimo"] * comparison["price"]

# Se ordena de mayor a menor cambio para ver quién gana y quién pierde unidades
display(comparison.sort_values("dif_unidades", ascending=False))

,demanda,price,intuitivo,optimo,dif_unidades,valor_intuitivo,valor_optimo
StockCode,,,,,,,
85099B,1175,2.08,823,1175,352,1711.84,2444.00
22086,1080,2.95,756,1080,324,2230.20,3186.00
22197,1809,0.85,1266,1554,288,1076.10,1320.90
23084,959,2.08,672,959,287,1397.76,1994.72
84879,860,1.69,601,860,259,1015.69,1453.40
21915,672,1.25,470,672,202,587.50,840.00
22578,693,0.29,485,139,-346,140.65,40.31
22577,746,0.29,522,150,-372,151.38,43.50
22952,765,0.55,535,153,-382,294.25,84.15


In [13]:
# Valor total de cada solución
v_int = float(comparison["valor_intuitivo"].sum())
v_opt = float(comparison["valor_optimo"].sum())
# Mejora absoluta (en £) y relativa (frente a la regla intuitiva)
improvement = v_opt - v_int
improvement_pct = improvement / v_int if v_int else np.nan

# Se informa siempre en £ y en porcentaje: es valor potencial de venta, no utilidad
print("Valor intuitivo (£):", round(v_int, 2))
print("Valor óptimo (£):", round(v_opt, 2))
print("Mejora absoluta (£):", round(improvement, 2))
print("Mejora porcentual:", f"{improvement_pct:.2%}")

Valor intuitivo (£): 8853.9
Valor óptimo (£): 11478.03
Mejora absoluta (£): 2624.13
Mejora porcentual: 29.64%


In [14]:
# Se reordena la tabla en formato largo para graficar los dos métodos lado a lado
long = (
    comparison[["intuitivo","optimo"]]
    .reset_index()
    .melt(id_vars="StockCode", var_name="metodo", value_name="unidades")
)

# Barras agrupadas: unidades asignadas por SKU, regla intuitiva frente a óptimo
fig = px.bar(
    long,
    x="StockCode",
    y="unidades",
    color="metodo",
    barmode="group",
    title="Asignación intuitiva vs óptima"
)
fig.show()

### Pregunta 7
¿Qué SKU ganan y pierden unidades, y cuál es la lógica económica de ese cambio?

**Respuesta:**

*   Ganan unidades: SKUs de mayor precio y alta demanda (ej. 85099B, 22086, 23084, 84879).
*   Pierden unidades: SKUs de menor precio (ej. 84077, 22952, 22577, 22578).
*   Lógica económica: El optimizador reasigna la capacidad limitada desde los productos de menor valor hacia los de mayor valor para maximizar el ingreso total, pero se detiene en el min_servicio para no desabastecer por completo a los productos de menor valor.


# 9. Verificación de restricciones

In [15]:
# Holgura = límite − uso: cuánto sobra en cada regla; 0 significa restricción activa
checks = pd.DataFrame(index=parametros.index)
checks["x"] = opt["asignacion_optima"]
checks["demanda"] = parametros["demanda"]
checks["min_servicio"] = parametros["min_servicio"]
checks["holgura_demanda"] = checks["demanda"] - checks["x"]
checks["holgura_minimo"] = checks["x"] - checks["min_servicio"]
checks["holgura_concentracion"] = limite_concentracion - checks["x"]
display(checks)
# Verificaciones: no pasar la capacidad y ninguna holgura negativa (sería una regla violada)
assert checks["x"].sum() <= capacidad_total
assert (checks[["holgura_demanda","holgura_minimo","holgura_concentracion"]] >= 0).all().all()
# Resumen: capacidad usada y si la concentración es vinculante (holgura 0 en algún SKU)
print("Capacidad usada:", int(checks["x"].sum()), "/", capacidad_total)
print("Límite de concentración vinculante:", bool((checks["holgura_concentracion"] == 0).any()))

,x,demanda,min_servicio,holgura_demanda,holgura_minimo,holgura_concentracion
StockCode,,,,,,
22197,1554,1809,362,255,1192,891
84077,245,1223,245,978,0,2200
85099B,1175,1175,235,0,940,1270
22086,1080,1080,216,0,864,1365
23084,959,959,192,0,767,1486
84879,860,860,172,0,688,1585
22952,153,765,153,612,0,2292
22577,150,746,150,596,0,2295
22578,139,693,139,554,0,2306


Capacidad usada: 6987 / 6987
Límite de concentración vinculante: False


### Pregunta 8
¿La concentración del 35 % es vinculante en el escenario base? Sustente con las holguras.

**Respuesta:** No, no es vinculante. El límite de concentración es de 2,445 unidades (35% de 6,987). La asignación máxima a un solo SKU es de 1,554 unidades (para el 22197), dejando una holgura de 891 unidades. Como todas las holguras de concentración son mayores a cero, la restricción no está activa (no limita la solución).

# 10. Sensibilidad a la capacidad

In [16]:
# Resuelve el mismo modelo con otros supuestos: devuelve el estado, el valor óptimo, el valor de la línea base y la asignación
def solve_scenario(params, capacity_ratio=0.70, min_service_ratio=0.20,
                   max_conc_ratio=0.35):
    # Se recalculan la capacidad y el tope de concentración del escenario
    total = int(params["demanda"].sum())
    capacity = int(np.floor(capacity_ratio * total))
    max_conc = int(np.floor(max_conc_ratio * capacity))
    p = params.copy()
    # Servicio mínimo del escenario (al menos una unidad por SKU)
    p["min_scn"] = np.maximum(np.ceil(min_service_ratio * p["demanda"]).astype(int), 1)
    # Si los mínimos no caben, el escenario es infactible y no se resuelve
    if p["min_scn"].sum() > capacity or (p["min_scn"] > max_conc).any():
        return {"status":"Infeasible", "capacity":capacity, "optimal_value":np.nan,
                "baseline_value":np.nan, "uplift_pct":np.nan, "allocation":None}
    # Mismo modelo entero del laboratorio, con los límites del escenario
    m = pl.LpProblem("Escenario", pl.LpMaximize)
    z = {sku: pl.LpVariable(f"z_{i}", lowBound=0, cat=pl.LpInteger)
         for i,sku in enumerate(p.index)}
    m += pl.lpSum(p.loc[sku,"price"]*z[sku] for sku in p.index)
    m += pl.lpSum(z.values()) <= capacity
    for sku in p.index:
        m += z[sku] <= int(p.loc[sku,"demanda"])
        m += z[sku] >= int(p.loc[sku,"min_scn"])
        m += z[sku] <= max_conc
    # Se resuelve y se lee el estado antes de usar los valores
    status = pl.LpStatus[m.solve(pl.PULP_CBC_CMD(msg=False))]
    if status != "Optimal":
        return {"status":status,"capacity":capacity,"optimal_value":np.nan,
                "baseline_value":np.nan,"uplift_pct":np.nan,"allocation":None}
    # Asignación óptima y línea base proporcional bajo las mismas reglas, para comparar con justicia
    allocation = pd.Series({sku:int(round(z[sku].value())) for sku in p.index})
    baseline_scn = proportional_allocation(p, capacity, min_col="min_scn", max_conc=max_conc)
    optimal_value = float((allocation*p["price"]).sum())
    baseline_value = float((baseline_scn*p["price"]).sum())
    return {"status":status,"capacity":capacity,"optimal_value":optimal_value,
            "baseline_value":baseline_value,
            "uplift_pct":100*(optimal_value/baseline_value-1),
            "allocation":allocation.to_dict()}

# Se resuelven los tres escenarios de capacidad: 60 %, 70 % y 85 % de la demanda
scenarios = [solve_scenario(parametros, capacity_ratio=r) for r in [0.60,0.70,0.85]]
# Se arma la tabla de resultados por escenario
scenario_df = pd.DataFrame([
    {"capacity_ratio": ratio, **{k:v for k,v in result.items() if k != "allocation"}}
    for ratio, result in zip([0.60,0.70,0.85], scenarios)
])
# solve_scenario devuelve uplift_pct como 40.07; Plotly necesita 0.4007 para el formato %.
scenario_df["improvement_pct"] = scenario_df["uplift_pct"] / 100
display(scenario_df)

,capacity_ratio,status,capacity,optimal_value,baseline_value,uplift_pct,improvement_pct
0,0.60,Optimal,5989,10629.73,7588.60,40.074981,0.400750
1,0.70,Optimal,6987,11478.03,8853.90,29.638126,0.296381
2,0.85,Optimal,8484,12214.08,10751.27,13.605927,0.136059


In [17]:
# Gráfico de la ventaja de optimizar según la capacidad disponible (60 %, 70 % y 85 %)
fig = px.bar(
    scenario_df,
    x="capacity_ratio",
    y="improvement_pct",
    text=scenario_df["improvement_pct"].map(lambda x: f"{x:.1%}" if pd.notna(x) else "NA"),
    title="Ventaja de optimizar según capacidad disponible",
    labels={"capacity_ratio":"Capacidad / demanda", "improvement_pct":"Mejora vs intuitivo"}
)
fig.show()

### Pregunta 9
¿Por qué la mejora porcentual de PuLP cambia cuando la capacidad pasa de 60 % a 85 %?

**Respuesta:** La mejora porcentual disminuye (de 40.1% a 13.6%) debido a los rendimientos marginales decrecientes. Con 60% de capacidad, la restricción es muy severa y el optimizador brilla al elegir solo los productos más rentables, mientras que la regla intuitiva desperdicia capacidad en productos de bajo valor. Con 85% de capacidad, hay suficiente espacio para satisfacer casi toda la demanda de todos los productos, por lo que la regla intuitiva (proporcional) ya se acerca mucho a la solución óptima, reduciendo la ventaja relativa de PuLP.

# 11. Decisión y recomendación

### Pregunta 10
¿Recomendaría operar con la asignación PuLP? Indique una condición que podría cambiar la respuesta.

**Respuesta:** Sí, se recomienda operar con la asignación de PuLP, ya que genera un valor potencial de £11,478.03 frente a £8,853.90 de la regla intuitiva (una mejora del 29.6%), respetando todas las restricciones de negocio.
Condición que cambiaría la respuesta: Si el costo de mantener inventario (holding cost) o el riesgo de obsolescencia de los productos de alto valor asignados fuera muy superior a lo estimado, o si la demanda real fuera significativamente menor a la pronosticada.

# 12. Retos individuales de aplicación

La parte guiada ya formuló y verificó el modelo. Resuelva los tres retos en su notebook personal. Son **10 preguntas** en total (4 + 3 + 3). En cada respuesta distinga resultados calculados, supuestos y recomendación. No altere las transacciones para producir una mejora mayor.

## Reto 1 — ¿Conviene ampliar la capacidad? (4 preguntas)

**Caso.** Operaciones puede aumentar la capacidad semanal, pero debe pagar por ella. Compare 60 %, 70 % y 85 % de la demanda total. Mantenga las demás reglas.

**R1.1.** Construya una tabla con capacidad, valor intuitivo, valor PuLP y mejora porcentual para los tres escenarios. ¿Qué representa cada cifra?

**R1.2.** Calcule el valor marginal por unidad adicional de capacidad entre 60 %→70 % y 70 %→85 %. ¿Por qué no son iguales?

**R1.3.** Si una unidad adicional de capacidad cuesta £0,65 por semana, ¿cuál ampliación conviene respecto del escenario de 60 %? Muestre valor incremental, costo incremental y saldo neto.

**R1.4.** Recomiende a operaciones una capacidad e indique una limitación de su cálculo.

In [22]:
# Reto 1: construya aquí su tabla y los cálculos incrementales.
# Sugerencia: use scenarios y capacidad_total; no modifique parametros.

R1.1. Construya una tabla con capacidad, valor intuitivo, valor PuLP y mejora porcentual para los tres escenarios. ¿Qué representa cada cifra?
# Función auxiliar para resolver los escenarios (incluye cálculo de línea base intuitiva)
def solve_scenario(params, capacity_ratio=0.70, min_service_ratio=0.20, max_conc_ratio=0.35):
    total = int(params['demanda'].sum())
    capacity = int(np.floor(capacity_ratio * total))
    max_conc = int(np.floor(max_conc_ratio * capacity))
    p = params.copy()
    p['min_scn'] = np.maximum(np.ceil(min_service_ratio * p['demanda']).astype(int), 1)

    if p['min_scn'].sum() > capacity or (p['min_scn'] > max_conc).any():
        return {"status": "Infeasible", "capacity": capacity, "optimal_value": np.nan, "baseline_value": np.nan, "allocation": None}

    m = pl.LpProblem("Escenario", pl.LpMaximize)
    z = {sku: pl.LpVariable(f"z_{i}", lowBound=0, cat=pl.LpInteger) for i, sku in enumerate(p.index)}
    m += pl.lpSum(p.loc[sku, 'price'] * z[sku] for sku in p.index)
    m += pl.lpSum(z.values()) <= capacity
    for sku in p.index:
        m += z[sku] <= int(p.loc[sku, 'demanda'])
        m += z[sku] >= int(p.loc[sku, 'min_scn'])
        m += z[sku] <= max_conc

    status = pl.LpStatus[m.solve(pl.PULP_CBC_CMD(msg=False))]
    if status != "Optimal":
        return {"status": status, "capacity": capacity, "optimal_value": np.nan, "baseline_value": np.nan, "allocation": None}

    allocation = pd.Series({sku: int(round(z[sku].value())) for sku in p.index})
    optimal_value = float((allocation * p['price']).sum())

    # Cálculo de la línea base intuitiva para este escenario específico
    upper = p['demanda'].astype(int).copy().clip(upper=int(max_conc))
    lower = p['min_scn'].astype(int).copy()
    alloc = lower.copy()
    remaining = int(capacity - alloc.sum())
    weights = p['demanda'].astype(float)

    while remaining > 0:
        room = upper - alloc
        active = room > 0
        if not active.any(): break
        quota = remaining * weights[active] / weights[active].sum()
        add = np.minimum(np.floor(quota).astype(int), room[active])
        if add.sum() == 0:
            order = (quota - np.floor(quota)).sort_values(ascending=False).index
            for sku in order:
                if remaining == 0: break
                if room.loc[sku] > 0:
                    alloc.loc[sku] += 1
                    remaining -= 1
        else:
            alloc.loc[active] += add
            remaining -= int(add.sum())

    baseline_value = float((alloc * p['price']).sum())
    uplift = 100 * (optimal_value / baseline_value - 1) if baseline_value > 0 else np.nan

    return {
        "status": status, "capacity": capacity, "optimal_value": optimal_value,
        "baseline_value": baseline_value, "uplift_pct": uplift, "allocation": allocation
    }

# --- RETO 1: Tabla de escenarios de capacidad ---
reto1_data = []
for ratio in [0.60, 0.70, 0.85]:
    res = solve_scenario(parametros, capacity_ratio=ratio, min_service_ratio=0.20, max_conc_ratio=0.35)
    reto1_data.append({
        "Capacidad (%)": f"{int(ratio*100)}%",
        "Capacidad (unid)": res['capacity'],
        "Valor Intuitivo (£)": round(res['baseline_value'], 2),
        "Valor PuLP (£)": round(res['optimal_value'], 2),
        "Mejora (%)": f"{res['uplift_pct']:.2f}%"
    })

df_reto1 = pd.DataFrame(reto1_data)
print("=== RETO 1: Escenarios de Capacidad ===")
display(df_reto1)

# Cálculos marginales R1.2 y R1.3
val_60 = df_reto1.loc[0, "Valor PuLP (£)"]
val_70 = df_reto1.loc[1, "Valor PuLP (£)"]
val_85 = df_reto1.loc[2, "Valor PuLP (£)"]
cap_60 = df_reto1.loc[0, "Capacidad (unid)"]
cap_70 = df_reto1.loc[1, "Capacidad (unid)"]
cap_85 = df_reto1.loc[2, "Capacidad (unid)"]

marginal_60_70 = (val_70 - val_60) / (cap_70 - cap_60)
marginal_70_85 = (val_85 - val_70) / (cap_85 - cap_70)

print(f"\nValor marginal 60%→70%: £{marginal_60_70:.2f} por unidad")
print(f"Valor marginal 70%→85%: £{marginal_70_85:.2f} por unidad")

costo_unitario = 0.65
saldo_60_70 = (val_70 - val_60) - ((cap_70 - cap_60) * costo_unitario)
saldo_70_85 = (val_85 - val_70) - ((cap_85 - cap_70) * costo_unitario)

print(f"\nSaldo neto al ampliar de 60% a 70%: £{saldo_60_70:.2f} (Conviene: {saldo_60_70 > 0})")
print(f"Saldo neto al ampliar de 70% a 85%: £{saldo_70_85:.2f} (Conviene: {saldo_70_85 > 0})")

Object `cifra` not found.
=== RETO 1: Escenarios de Capacidad ===


,Capacidad (%),Capacidad (unid),Valor Intuitivo (£),Valor PuLP (£),Mejora (%)
0,60%,5989,7588.60,10629.73,40.07%
1,70%,6987,8853.90,11478.03,29.64%
2,85%,8484,10751.27,12214.08,13.61%



Valor marginal 60%→70%: £0.85 por unidad
Valor marginal 70%→85%: £0.49 por unidad

Saldo neto al ampliar de 60% a 70%: £199.60 (Conviene: True)
Saldo neto al ampliar de 70% a 85%: £-237.00 (Conviene: False)


R1.2. Valor marginal por unidad adicional:
*   60% → 70%: (11,478.03 - 10,629.73) / (6,987 - 5,989) = 848.30 / 998 ≈ £0.85 por unidad.
*   70% → 85%: (12,214.08 - 11,478.03) / (8,484 - 6,987) = 736.05 / 1,497 ≈ £0.49 por unidad.

¿Por qué no son iguales? Por los rendimientos marginales decrecientes: al haber más capacidad, el optimizador se ve obligado a asignar unidades a productos de menor precio, ya que la demanda de los más rentables ya está casi satisfecha.

R1.3. Evaluación con costo de £0.65 por unidad:
*   Ampliar de 60% a 70%: Valor marginal (£0.85) > Costo (£0.65). Saldo neto = 848.30 - (998 × 0.65) = +£199.60. Sí conviene.
*   Ampliar de 70% a 85%: Valor marginal (£0.49) < Costo (£0.65). Saldo neto = 736.05 - (1,497 × 0.65) = -£237.00. No conviene.



R1.4. Recomendación y limitación:
Recomiendo a operaciones ampliar la capacidad al 70%.

Limitación: El cálculo asume que todas las unidades asignadas se venderán (valor potencial), ignorando costos de almacenamiento, costos de pedido y la variabilidad real de la demanda.


## Reto 2 — Servicio mínimo y costo de la equidad (3 preguntas)

**Caso.** Comercial propone elevar el servicio mínimo para evitar productos casi desabastecidos. Evalúe 20 %, 30 % y 50 % con capacidad 70 %.

**R2.1.** Resuelva los tres escenarios y muestre valor óptimo y unidades por SKU. ¿Cuál sigue siendo factible?

**R2.2.** Calcule cuánto valor potencial se sacrifica al pasar de 20 % a 30 % y de 20 % a 50 %. Identifique qué SKU cambian más.

**R2.3.** Recomiende una política de servicio. Explique qué indicador operativo faltaría para decidirla en una empresa real.

In [23]:
# Reto 2: complete la tabla de servicio y una comparación por SKU.
# Sugerencia: solve_scenario(parametros, min_service_ratio=...)

reto2_data = []
print("=== RETO 2: Escenarios de Servicio Mínimo ===")

for min_sr in [0.20, 0.30, 0.50]:
    res = solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=min_sr, max_conc_ratio=0.35)
    reto2_data.append({
        "Servicio Mínimo": f"{int(min_sr*100)}%",
        "Estado": res['status'],
        "Valor Óptimo (£)": round(res['optimal_value'], 2) if res['status'] == "Optimal" else "N/A"
    })

df_reto2 = pd.DataFrame(reto2_data)
display(df_reto2)

# Detalle de asignación y sacrificio de valor
print("\n--- Detalle de Asignación por SKU y Sacrificio de Valor ---")
res_20 = solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.20, max_conc_ratio=0.35)
res_30 = solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.30, max_conc_ratio=0.35)
res_50 = solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.50, max_conc_ratio=0.35)

print(f"Sacrificio de 20% a 30%: £{res_20['optimal_value'] - res_30['optimal_value']:.2f}")
print(f"Sacrificio de 20% a 50%: £{res_20['optimal_value'] - res_50['optimal_value']:.2f}")

# Mostrar cambios en SKUs específicos
print("\nComparativa de asignación (SKU que más cambian):")
comparativa = pd.DataFrame({
    "Servicio 20%": res_20['allocation'],
    "Servicio 30%": res_30['allocation'],
    "Servicio 50%": res_50['allocation']
})
comparativa['Dif 20% a 50%'] = comparativa['Servicio 50%'] - comparativa['Servicio 20%']
display(comparativa.sort_values('Dif 20% a 50%'))

=== RETO 2: Escenarios de Servicio Mínimo ===


,Servicio Mínimo,Estado,Valor Óptimo (£)
0,20%,Optimal,11478.03
1,30%,Optimal,11306.53
2,50%,Optimal,10791.63



--- Detalle de Asignación por SKU y Sacrificio de Valor ---
Sacrificio de 20% a 30%: £171.50
Sacrificio de 20% a 50%: £686.40

Comparativa de asignación (SKU que más cambian):


,Servicio 20%,Servicio 30%,Servicio 50%,Dif 20% a 50%
22197,1554,1212,905,-649
21915,672,672,336,-336
84879,860,860,817,-43
22086,1080,1080,1080,0
23084,959,959,959,0
85099B,1175,1175,1175,0
22578,139,208,347,208
22577,150,224,373,223
22952,153,230,383,230
84077,245,367,612,367


R2.2. Sacrificio de valor potencial:
* De 20% a 30%: 11,478.03 - 11,306.53 = £171.50.
* De 20% a 50%: 11,478.03 - 10,791.63 = £686.40.

SKUs que más cambian: Los de menor precio (ej. 84077, 22577, 22578, 22952), cuyas asignaciones aumentan forzosamente para cumplir el mínimo, restando capacidad a los de mayor valor (ej. 22197 baja de 1,554 a 905 unidades).

R2.3. Recomendación e indicador faltante:
Recomiendo mantener una política de servicio mínimo del 20% o 30%. Exigir 50% sacrifica demasiado valor (£686.40) por "equidad".

Indicador faltante: El costo de quiebre de stock (stockout cost) o el margen de contribución real (no solo el precio de venta), además de los tiempos de reposición (lead time).

## Reto 3 — ¿La concentración limita de verdad? (3 preguntas)

**Caso.** Riesgos propone limitar cada SKU al 20 %, 25 %, 35 % o 50 % de la capacidad. Mantenga el escenario de capacidad 70 % y servicio mínimo 20 %.

**R3.1.** Resuelva las cuatro políticas. Presente valor óptimo, asignación y factibilidad.

**R3.2.** Indique en qué políticas la restricción cambia la solución y calcule el valor perdido frente a 35 %. ¿Qué demuestra el escenario de 50 %?

**R3.3.** Recomiende un límite de concentración y explique por qué el porcentaje por sí solo no mide todo el riesgo de inventario.

In [25]:
# Reto 3: complete la tabla de concentración y compare por SKU.
# Sugerencia: solve_scenario(parametros, max_conc_ratio=...)

reto3_data = []
print("=== RETO 3: Escenarios de Concentración Máxima ===")

for max_cr in [0.20, 0.25, 0.35, 0.50]:
    res = solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.20, max_conc_ratio=max_cr)
    # Se corrige .values() por .values al ser una propiedad de la Serie de pandas
    max_asignado = max(res['allocation'].values) if res['allocation'] is not None else 0
    reto3_data.append({
        "Concentración Máx": f"{int(max_cr*100)}%",
        "Estado": res['status'],
        "Valor Óptimo (£)": round(res['optimal_value'], 2) if res['status'] == "Optimal" else "N/A",
        "Asignación Máx (SKU 22197)": max_asignado
    })

df_reto3 = pd.DataFrame(reto3_data)
display(df_reto3)

# Cálculo de valor perdido
res_35 = solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.20, max_conc_ratio=0.35)
res_20 = solve_scenario(parametros, capacity_ratio=0.70, min_service_ratio=0.20, max_conc_ratio=0.20)

valor_perdido = res_35['optimal_value'] - res_20['optimal_value']
print(f"\nValor perdido al limitar concentración al 20% (vs 35%): £{valor_perdido:.2f}")
print("Nota: A partir del 25%, la restricción de concentración deja de ser vinculante (no cambia la solución óptima).")

=== RETO 3: Escenarios de Concentración Máxima ===


,Concentración Máx,Estado,Valor Óptimo (£),Asignación Máx (SKU 22197)
0,20%,Optimal,11430.93,1397
1,25%,Optimal,11478.03,1554
2,35%,Optimal,11478.03,1554
3,50%,Optimal,11478.03,1554



Valor perdido al limitar concentración al 20% (vs 35%): £47.10
Nota: A partir del 25%, la restricción de concentración deja de ser vinculante (no cambia la solución óptima).


R3.1. Escenarios con capacidad 70% y servicio 20%:
* Concentración 20%: Valor óptimo = £11,430.93. Factible.
* Concentración 25%: Valor óptimo = £11,478.03. Factible.
* Concentración 35%: Valor óptimo = £11,478.03. Factible.
* Concentración 50%: Valor óptimo = £11,478.03. Factible.

R3.2. Políticas que cambian la solución y valor perdido:

La restricción solo cambia la solución en la política del 20% (donde el tope es 1,397 unidades, obligando a reducir el SKU 22197 de 1,554 a 1,397).
* Valor perdido frente al 35%: 11,478.03 - 11,430.93 = £47.10.
* ¿Qué demuestra el 50%? Demuestra que, a partir de cierto umbral (~22.3%), la restricción de concentración deja de ser vinculante y se vuelve irrelevante para la solución óptima.

R3.3. Recomendación y riesgo del porcentaje:

Recomiendo un límite de concentración del 25% o 35%.
Por qué el porcentaje no mide todo el riesgo: Un porcentaje fijo no considera la variabilidad de la demanda (volatilidad), el valor monetario absoluto del inventario, ni el riesgo de obsolescencia específico de cada SKU. Un 20% de un producto de alta rotación y bajo costo es mucho menos riesgoso que un 20% de un producto lento y caro.


# 13. Verificaciones finales

- Revise que cada solución factible respete capacidad, demanda, servicio y concentración.
- Indique si la regla intuitiva usa exactamente las mismas restricciones.
- Compruebe que las diez preguntas de los retos tengan respuesta numérica o una decisión justificada.
- Declare una limitación: ventas observadas ≠ demanda total; valor potencial ≠ utilidad.

# 14. Cierre ejecutivo y ticket de salida

Redacte máximo 12 líneas: decisión recomendada, capacidad, valor intuitivo y óptimo, mejora, costo de ampliar capacidad, política de servicio, concentración, principal riesgo y condición que cambiaría la recomendación.

**Ticket:** «Con ______ unidades de capacidad, PuLP genera £______ frente a £______ de la regla intuitiva. Recomiendo ______ porque ______. Esta conclusión cambiaría si ______».

«Con 6,987 unidades de capacidad, PuLP genera £11,478.03 frente a £8,853.90 de la regla intuitiva. Recomiendo operar con el modelo de PuLP al 70% de capacidad porque maximiza el valor potencial en un 29.6% respetando los niveles mínimos de servicio. Esta conclusión cambiaría si los costos de mantenimiento de inventario o la volatilidad de la demanda fueran significativamente mayores a los supuestos actuales».

# 15. Entrega y referencias

Suba el notebook ejecutado a `semana11/` de su repositorio personal y comparta el enlace de Colab. Incluya los tres retos y las diez respuestas individuales. Declare el uso de IA.

Chen, D. (2012). *Online Retail II* [dataset]. UCI Machine Learning Repository. DOI: 10.24432/C5CG6D.
Sharda, R., Delen, D. y Turban, E. *Business Intelligence, Analytics, Data Science and AI*.
Winston, W. L. *Operations Research: Applications and Algorithms*.